<a href="https://colab.research.google.com/github/Gema-Villanueva/Brecha-Digital-2/blob/main/Dataset__Maestro_Brecha_Digital.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dataset construido a partir de datos oficiales del INE y del Ministerio de Educación integrando variables socioeconómicas y de acceso digital.
Se realiza un proceso de limpieza de datos que incluye la normalización de nombres, conversión de formatos numéricos y unificación de datasets.

In [1]:
!pip install -q pandas numpy

In [2]:
#Importamos las librerías necesarias para análisis de datos
import pandas as pd
import numpy as np

In [3]:
#Convertimos los números de formato español (con coma) a formato python (con punto)
def to_float_es(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x == "":
        return np.nan

    if "," in x:
        x = x.replace(".", "").replace(",", ".")
    else:
        x = x.replace(".", "")

    try:
        return float(x)
    except:
        return np.nan


#limpiamos y unificamos los nombres de las comunidades autónomas
def clean_name(name):
    if pd.isna(name):
        return None

    name = str(name).strip()
    name = name.replace("\xa0", " ").strip()

    # quitar códigos delante tipo "01 Andalucía"
    name = pd.Series([name]).str.replace(r"^\d+\s*", "", regex=True).iloc[0]

    cambios = {
        "Asturias, Principado de": "Asturias",
        "Principado de Asturias": "Asturias",
        "Asturias (Principado de)": "Asturias",

        "Balears, Illes": "Baleares",
        "Illes Balears": "Baleares",
        "Balears (Illes)": "Baleares",

        "Madrid, Comunidad de": "Madrid",
        "Comunidad de Madrid": "Madrid",
        "Madrid (Comunidad de)": "Madrid",

        "Murcia, Región de": "Murcia",
        "Región de Murcia": "Murcia",
        "Murcia (Región de)": "Murcia",

        "Navarra, Comunidad Foral de": "Navarra",
        "Comunidad Foral de Navarra": "Navarra",
        "Navarra (Comunidad Foral de)": "Navarra",

        "Rioja, La": "La Rioja",
        "La Rioja": "La Rioja",
        "Rioja (La)": "La Rioja",

        "Comunitat Valenciana": "Comunidad Valenciana",
        "Castilla - La Mancha": "Castilla-La Mancha",

        "Total nacional": "Total Nacional",
        "TOTAL": "Total Nacional"
    }

    return cambios.get(name, name)

In [4]:
CCAA_17 = [
    "Andalucía",
    "Aragón",
    "Asturias",
    "Baleares",
    "Canarias",
    "Cantabria",
    "Castilla y León",
    "Castilla-La Mancha",
    "Cataluña",
    "Comunidad Valenciana",
    "Extremadura",
    "Galicia",
    "Madrid",
    "Murcia",
    "Navarra",
    "País Vasco",
    "La Rioja"
]

In [5]:
#cargamos datasets desde fuentes oficiales INE y Ministerio
internet_raw = pd.read_csv("https://www.ine.es/jaxi/files/tpx/csv_bd/50039.csv", sep="\t")
renta_raw = pd.read_csv("https://www.ine.es/jaxiT3/files/t/csv_bd/68338.csv", sep="\t")
arope_raw = pd.read_csv("https://www.ine.es/jaxiT3/files/t/csv_bdsc/60264.csv", sep=";")
paro_raw = pd.read_csv("https://www.ine.es/jaxiT3/files/t/csv_bdsc/67565.csv", sep=";")
orden_raw = pd.read_csv(
    "https://estadisticas.educacion.gob.es/EducaJaxiPx/files/_px/es/csv_bdsc/no-universitaria/centros/tic/series/l0/series_1_4.csv_bdsc?nocab=1",
    sep=";"
)

In [6]:
#Filtramos los datos de acceso a banda ancha para el año 2021
internet = internet_raw.copy()

internet["comunidad"] = internet["Comunidades Autónomas"].apply(clean_name)
internet["anio"] = pd.to_numeric(internet["Periodo"], errors="coerce")
internet["valor"] = internet["Total"].apply(to_float_es)

internet_2021 = (
    internet[internet["Tipo de equipamiento"] == "Viviendas con conexión de Banda Ancha"]
    .loc[:, ["comunidad", "anio", "valor"]]
    .rename(columns={"valor": "banda_ancha_pct"})
)

internet_2021 = internet_2021[
    (internet_2021["anio"] == 2021) &
    (internet_2021["comunidad"].isin(CCAA_17))
].copy()

internet_2021["anio"] = internet_2021["anio"].astype(int)

In [7]:
#Filtramos la renta media por comunidad autónoma
renta = renta_raw.copy()

renta["comunidad"] = renta["Comunidad autónoma"].apply(clean_name)
renta["anio"] = pd.to_numeric(renta["Periodo"], errors="coerce")
renta["valor"] = renta["Total"].apply(to_float_es)

renta_2021 = (
    renta[renta["Renta media y mediana"].astype(str).str.contains("renta media", case=False, na=False)]
    .loc[:, ["comunidad", "anio", "valor"]]
    .rename(columns={"valor": "renta_media"})
)

renta_2021 = renta_2021[
    (renta_2021["anio"] == 2021) &
    (renta_2021["comunidad"].isin(CCAA_17))
].copy()

renta_2021 = renta_2021.groupby("comunidad", as_index=False)["renta_media"].mean()
renta_2021["anio"] = 2021

In [8]:
#Filtramos el indicador de riesgo de pobreza y exclusión social
arope = arope_raw.copy()

col_comunidad_arope = [c for c in arope.columns if "Comunidades" in c or "comunidad" in c.lower()][0]
col_periodo_arope = [c for c in arope.columns if "Periodo" in c or "periodo" in c.lower()][0]
col_valor_arope = "Total" if "Total" in arope.columns else arope.columns[-1]

arope["comunidad"] = arope[col_comunidad_arope].apply(clean_name)
arope["anio"] = pd.to_numeric(arope[col_periodo_arope], errors="coerce")
arope["valor"] = arope[col_valor_arope].apply(to_float_es)

posibles_indicadores = [
    c for c in arope.columns
    if c not in [col_comunidad_arope, col_periodo_arope, col_valor_arope, "comunidad", "anio", "valor"]
]
indicador_col = posibles_indicadores[0]

arope_2021 = (
    arope[arope[indicador_col].astype(str).str.contains("riesgo de pobreza|arope", case=False, na=False)]
    .loc[:, ["comunidad", "anio", "valor"]]
    .rename(columns={"valor": "arope_pct"})
)

arope_2021 = arope_2021[
    (arope_2021["anio"] == 2021) &
    (arope_2021["comunidad"].isin(CCAA_17))
].copy()

arope_2021 = arope_2021.groupby("comunidad", as_index=False)["arope_pct"].mean()
arope_2021["anio"] = 2021

In [9]:
#Filtramos la tasa de desempleo
paro = paro_raw.copy()

col_comunidad_paro = [c for c in paro.columns if "Comunidades" in c or "comunidad" in c.lower()][0]
col_periodo_paro = [c for c in paro.columns if "Periodo" in c or "periodo" in c.lower()][0]
col_valor_paro = "Total" if "Total" in paro.columns else paro.columns[-1]

paro["comunidad"] = paro[col_comunidad_paro].apply(clean_name)
paro["valor"] = paro[col_valor_paro].apply(to_float_es)
paro["anio"] = paro[col_periodo_paro].astype(str).str.extract(r"(\d{4})").astype(float)

if "Sexo" in paro.columns:
    paro = paro[paro["Sexo"] == "Ambos sexos"].copy()

if "Edad" in paro.columns:
    paro = paro[paro["Edad"] == "Total"].copy()

paro_2021 = (
    paro[(paro["anio"] == 2021) & (paro["comunidad"].isin(CCAA_17))]
    .groupby("comunidad", as_index=False)["valor"].mean()
    .rename(columns={"valor": "paro_pct"})
)

paro_2021["anio"] = 2021

In [10]:
#Datos sobre disponibilidad de ordenadores en centros educativos
orden = orden_raw.copy()

orden["comunidad"] = orden["Comunidad autónoma"].apply(clean_name)
orden["valor"] = orden["Total"].apply(to_float_es)

orden_2021 = (
    orden[
        (orden["Titularidad"] == "TODOS LOS CENTROS") &
        (orden["periodo"] == "2020-2021") &
        (orden["comunidad"].isin(CCAA_17))
    ]
    .loc[:, ["comunidad", "valor"]]
    .rename(columns={"valor": "ordenadores_unidad"})
)

orden_2021 = orden_2021.groupby("comunidad", as_index=False)["ordenadores_unidad"].mean()

In [11]:
#Unimos todos los datasets en un único dataset maestro
df = internet_2021.merge(renta_2021, on=["comunidad", "anio"], how="inner")
df = df.merge(arope_2021, on=["comunidad", "anio"], how="inner")
df = df.merge(paro_2021, on=["comunidad", "anio"], how="inner")
df = df.merge(orden_2021, on="comunidad", how="left")

df = df[df["comunidad"].isin(CCAA_17)].copy()
df = df.sort_values("comunidad").reset_index(drop=True)

In [12]:
#Nos aseguramos que estén todas las comunidades autónomas
display(df)
print("Shape final:", df.shape)
print("Comunidades finales:", sorted(df["comunidad"].unique()))

,comunidad,anio,banda_ancha_pct,renta_media,arope_pct,paro_pct,ordenadores_unidad
0,Andalucía,2021,94.5,13752.5,35.50,21.86,5.7
1,Aragón,2021,96.3,18751.0,17.90,10.34,6.1
2,Asturias,2021,94.3,17384.5,23.35,12.35,8.1
3,Baleares,2021,97.3,16316.0,20.00,14.33,NaN
4,Canarias,2021,96.7,14528.5,33.10,23.48,7.9
5,Cantabria,2021,95.0,17511.5,18.40,11.27,NaN
6,Castilla y León,2021,95.6,17434.5,20.50,15.67,5.1
7,Castilla-La Mancha,2021,94.4,7794.5,29.95,11.62,5.6
8,Cataluña,2021,96.9,19877.5,18.55,11.70,10.2
9,Comunidad Valenciana,2021,96.8,15873.5,27.85,16.10,5.2


Shape final: (17, 7)
Comunidades finales: ['Andalucía', 'Aragón', 'Asturias', 'Baleares', 'Canarias', 'Cantabria', 'Castilla y León', 'Castilla-La Mancha', 'Cataluña', 'Comunidad Valenciana', 'Extremadura', 'Galicia', 'La Rioja', 'Madrid', 'Murcia', 'Navarra', 'País Vasco']


In [13]:
df.to_csv("dataset_maestro_2021.csv", index=False)
print("✅ Dataset guardado correctamente como dataset_maestro_2021.csv")

✅ Dataset guardado correctamente como dataset_maestro_2021.csv


Dataset construido a partir de datos oficiales del INE y del Ministerio de Educación integrando variables socioeconómicas y de acceso digital.
Se realiza un proceso de limpieza de datos que incluye la normalización de nombres, conversión de formatos numéricos y unificación de datasets.